In [ ]:
"""
check_recording_lengths.py
--------------------------
Reads all epoched .fif recordings for every participant / mode and
reports the usable recording length (total time across all epochs).

Directory structure assumed:
    ../data/out/subjects/<participant>/<mode>/<participant>_epoched.fif

Groups
------
    YAC  : 101-120
    OAC  : 202-221
    TCOA : 401-416
"""

import os
import mne
import pandas as pd
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────────────────

BASE_DIR = Path("../data/out/subjects")
MODES    = ["EC", "EO"]

# All participant IDs across the three groups
PARTICIPANTS = {
    "YAC" : [str(p) for p in range(101, 121)],   # 101 – 120
    "OAC" : [str(p) for p in range(201, 221)],   # 202 – 221
    "TCOA": [str(p) for p in range(401, 416)],   # 401 – 416
}

# ── Main ─────────────────────────────────────────────────────────────────────

records = []

for group, participants in PARTICIPANTS.items():
    for participant in participants:
        for mode in MODES:

            fif_path = BASE_DIR / participant / mode / f"{participant}_epoched.fif"

            if not fif_path.exists():
                print(f"  [MISSING]  {fif_path}")
                records.append({
                    "Group"       : group,
                    "Participant" : participant,
                    "Mode"        : mode,
                    "Status"      : "Missing",
                    "N_Epochs"    : None,
                    "Epoch_Duration_s"   : None,
                    "Total_Duration_s"   : None,
                    "Total_Duration_min" : None,
                    "Sfreq_Hz"    : None,
                    "N_Channels"  : None,
                    "N_Times"     : None,
                })
                continue

            try:
                # preload=False keeps memory usage low — we only need metadata
                epochs = mne.read_epochs(str(fif_path), preload=False, verbose=False)

                n_epochs          = len(epochs)
                epoch_duration_s  = epochs.times[-1] - epochs.times[0]
                total_duration_s  = n_epochs * epoch_duration_s
                total_duration_min = total_duration_s / 60.0
                sfreq             = epochs.info["sfreq"]
                n_channels        = len(epochs.ch_names)
                n_times           = len(epochs.times)

                print(
                    f"  [OK]  {group:4s}  {participant}  {mode}  "
                    f"| {n_epochs} epochs × {epoch_duration_s:.3f} s "
                    f"= {total_duration_s:.2f} s  ({total_duration_min:.2f} min)"
                )

                records.append({
                    "Group"              : group,
                    "Participant"        : participant,
                    "Mode"               : mode,
                    "Status"             : "OK",
                    "N_Epochs"           : n_epochs,
                    "Epoch_Duration_s"   : round(epoch_duration_s, 4),
                    "Total_Duration_s"   : round(total_duration_s, 4),
                    "Total_Duration_min" : round(total_duration_min, 4),
                    "Sfreq_Hz"           : sfreq,
                    "N_Channels"         : n_channels,
                    "N_Times"            : n_times,
                })

            except Exception as exc:
                print(f"  [ERROR]  {fif_path}  →  {exc}")
                records.append({
                    "Group"              : group,
                    "Participant"        : participant,
                    "Mode"               : mode,
                    "Status"             : f"Error: {exc}",
                    "N_Epochs"           : None,
                    "Epoch_Duration_s"   : None,
                    "Total_Duration_s"   : None,
                    "Total_Duration_min" : None,
                    "Sfreq_Hz"           : None,
                    "N_Channels"         : None,
                    "N_Times"            : None,
                })

# ── Summary table ─────────────────────────────────────────────────────────────

df = pd.DataFrame(records)

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

ok_df = df[df["Status"] == "OK"]

if not ok_df.empty:
    print("\nRecording duration statistics (seconds) — successfully loaded files:")
    print(ok_df.groupby(["Group", "Mode"])["Total_Duration_s"].describe().round(2).to_string())

    print("\nPer-participant summary:")
    display_cols = [
        "Group", "Participant", "Mode",
        "N_Epochs", "Epoch_Duration_s", "Total_Duration_s", "Total_Duration_min",
        "Sfreq_Hz", "N_Channels",
    ]
    print(ok_df[display_cols].to_string(index=False))

missing_df = df[df["Status"] != "OK"]
if not missing_df.empty:
    print(f"\nFiles not loaded ({len(missing_df)} total):")
    print(missing_df[["Group", "Participant", "Mode", "Status"]].to_string(index=False))

# ── Save to CSV ───────────────────────────────────────────────────────────────

out_csv = Path("recording_lengths.csv")
df.to_csv(out_csv, index=False)
print(f"\nResults saved to: {out_csv.resolve()}")

In [ ]:
# ── Average recording length across all participants ──────────────────────────

avg_overall = ok_df["Total_Duration_min"].mean()
print(f"\nOverall average recording length: {avg_overall:.2f} min")

avg_by_mode = ok_df.groupby("Mode")["Total_Duration_min"].mean()
print("\nAverage recording length by mode:")
print(avg_by_mode.round(2).to_string())

avg_by_group = ok_df.groupby("Group")["Total_Duration_min"].mean()
print("\nAverage recording length by group:")
print(avg_by_group.round(2).to_string())

avg_by_group_mode = ok_df.groupby(["Group", "Mode"])["Total_Duration_min"].mean()
print("\nAverage recording length by group and mode:")
print(avg_by_group_mode.round(2).to_string())

In [ ]:
"""
icc_reliability.py
------------------
Calculates ICC(2,1) — two-way random effects, single measures — to assess
the reliability/stability of correlation matrices across windows within
each brain state, for every participant, mode, and group.

Data structure
--------------
File:  <BASE_DIR>/<participant>/<mode>/<participant>_correlation_matrices.npy.npz
Keys:  (state, window) tuples  →  numpy array of shape (748,)
       748 = vectorised upper triangle of the 100×100 Schaefer parcellation

ICC logic
---------
For a given participant / mode / state:
  - Rows  = 748 edges (items)
  - Cols  = N windows assigned to that state (raters/repeated measures)
  - ICC(2,1) asks: across windows, how consistently is each edge ranked?
  - A high ICC (> 0.75) means the connectivity pattern is stable across
    the windows belonging to that state.

Outputs
-------
  icc_per_participant.csv   — one ICC per participant x mode x state
  icc_summary_by_group.csv  — mean +/- SD ICC aggregated by group and mode
"""

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────

BASE_DIR = Path("../data/out/subjects")
MODES    = ["EC", "EO"]

PARTICIPANTS = {
    "YAC" : [str(p) for p in range(101, 121)],   # 101 – 120
    "OAC" : [str(p) for p in range(202, 222)],   # 202 – 221
    "TCOA": [str(p) for p in range(401, 417)],   # 401 – 416
}

# ── ICC(2,1) implementation ───────────────────────────────────────────────────

def icc_2_1(data):
    """
    Compute ICC(2,1): two-way random effects, single measures (consistency).

    Parameters
    ----------
    data : np.ndarray, shape (n_items, n_raters)
        Rows = items (edges), Columns = raters (windows).

    Returns
    -------
    float : ICC value in [-1, 1]. Returns NaN if computation is not possible.
    """
    n, k = data.shape  # n = edges, k = windows

    if k < 2:
        return np.nan

    grand_mean = data.mean()
    row_means  = data.mean(axis=1)
    col_means  = data.mean(axis=0)

    ss_rows  = k * np.sum((row_means - grand_mean) ** 2)
    ss_cols  = n * np.sum((col_means - grand_mean) ** 2)
    ss_total = np.sum((data - grand_mean) ** 2)
    ss_error = ss_total - ss_rows - ss_cols

    df_rows  = n - 1
    df_cols  = k - 1
    df_error = df_rows * df_cols

    if df_error == 0:
        return np.nan

    ms_rows  = ss_rows  / df_rows
    ms_cols  = ss_cols  / df_cols
    ms_error = ss_error / df_error

    icc = (ms_rows - ms_error) / (ms_rows + (k - 1) * ms_error + k * (ms_cols - ms_error) / n)

    return float(icc)


def interpret_icc(icc):
    """Koo & Mae (2016) benchmarks for ICC interpretation."""
    if np.isnan(icc):
        return "N/A"
    elif icc < 0.50:
        return "Poor"
    elif icc < 0.75:
        return "Moderate"
    elif icc < 0.90:
        return "Good"
    else:
        return "Excellent"


def parse_state_window(key):
    """
    Extract (state, window) integers from an npz key.
    Handles formats: "(0, 1)", "0_1", tuple/array of two ints.
    Returns (None, None) if the key cannot be parsed.
    """
    if isinstance(key, str):
        # Format "(state, window)" or "state, window"
        match = re.search(r'\(?\s*(\d+)\s*,\s*(\d+)\s*\)?', key)
        if match:
            return int(match.group(1)), int(match.group(2))
        # Format "state_window"
        parts = key.split('_')
        if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
            return int(parts[0]), int(parts[1])
    elif hasattr(key, '__len__') and len(key) == 2:
        return int(key[0]), int(key[1])
    elif isinstance(key, (int, np.integer)):
        return 0, int(key)
    return None, None


def extract_numeric_vector(vec):
    """
    Return a float array from vec regardless of whether it was stored as
    raw floats or as labelled strings like '[Network]: R1 - 0.15 - R2'.
    Returns None if extraction fails.
    """
    if vec.dtype.kind in ('U', 'S', 'O'):
        try:
            nums = [float(str(entry).split(' - ')[1]) for entry in vec]
            return np.array(nums, dtype=float)
        except Exception:
            return None
    else:
        return vec.astype(float)


# ── Main loop ─────────────────────────────────────────────────────────────────

records       = []
first_file_ok = False   # flag so diagnostic only prints once

for group, participants in PARTICIPANTS.items():
    for participant in participants:
        for mode in MODES:

            npz_path = BASE_DIR / participant / mode / f"{participant}_correlation_matrices.npy.npz"

            if not npz_path.exists():
                print(f"  [MISSING]  {npz_path}")
                continue

            try:
                data = np.load(npz_path, allow_pickle=True)
            except Exception as e:
                print(f"  [ERROR loading]  {npz_path}  →  {e}")
                continue

            # ── One-time diagnostic to confirm key format ──────────────────
            if not first_file_ok:
                print(f"\n{'='*60}")
                print(f"DIAGNOSTIC — first file loaded: {npz_path}")
                print(f"  Total keys : {len(data.files)}")
                print(f"  First 5    : {data.files[:5]}")
                print(f"  Key type   : {type(data.files[0])}")
                sample_val = data[data.files[0]]
                print(f"  Value shape: {sample_val.shape}")
                print(f"  Value dtype: {sample_val.dtype}")
                print(f"  First entry: {sample_val[0]}")
                print(f"{'='*60}\n")
                first_file_ok = True

            # ── Parse keys and group windows by state ──────────────────────
            state_windows = {}
            n_skipped = 0

            for key in data.files:
                state, window = parse_state_window(key)

                if state is None:
                    n_skipped += 1
                    continue

                vec = extract_numeric_vector(data[key])
                if vec is None:
                    n_skipped += 1
                    continue

                if vec.ndim > 1:
                    vec = vec.flatten()

                state_windows.setdefault(state, []).append(vec)

            if n_skipped:
                print(f"  [WARN] {participant}/{mode}: {n_skipped} keys skipped (could not parse)")

            if not state_windows:
                print(f"  [WARN] {participant}/{mode}: no valid states found — check key format above")
                continue

            # ── Compute ICC per state ──────────────────────────────────────
            for state, vecs in state_windows.items():
                n_windows = len(vecs)

                if n_windows < 2:
                    icc_val = np.nan
                    interp  = "N/A (only 1 window)"
                else:
                    try:
                        matrix  = np.column_stack(vecs)
                        icc_val = icc_2_1(matrix)
                        interp  = interpret_icc(icc_val)
                    except Exception as e:
                        icc_val = np.nan
                        interp  = f"Error: {e}"

                records.append({
                    "Group"          : group,
                    "Participant"    : participant,
                    "Mode"           : mode,
                    "State"          : state,
                    "N_Windows"      : n_windows,
                    "ICC_2_1"        : round(icc_val, 4) if not np.isnan(icc_val) else np.nan,
                    "Interpretation" : interp,
                })

            valid_iccs = [r["ICC_2_1"] for r in records
                          if r["Participant"] == participant and r["Mode"] == mode
                          and not np.isnan(r["ICC_2_1"])]
            mean_icc   = np.mean(valid_iccs) if valid_iccs else np.nan
            mean_str   = f"{mean_icc:.3f}" if not np.isnan(mean_icc) else "N/A"
            print(f"  {group:4s}  {participant}  {mode}  | {len(state_windows)} states  | mean ICC = {mean_str}")

# ── Build DataFrame ───────────────────────────────────────────────────────────

if not records:
    print("\n[ERROR] No records were collected. Possible causes:")
    print("  1. No .npz files were found — check BASE_DIR path.")
    print("  2. Keys could not be parsed — check DIAGNOSTIC output above.")
    print("  3. All files failed to load — check file permissions.")
    raise SystemExit(1)

df = pd.DataFrame(records)

# ── Save per-participant results ──────────────────────────────────────────────

out_dir    = Path(".")
per_p_path = out_dir / "icc_per_participant.csv"
df.to_csv(per_p_path, index=False)
print(f"\nPer-participant ICC saved to: {per_p_path.resolve()}")

# ── Summary by group and mode ─────────────────────────────────────────────────

valid_df = df[df["ICC_2_1"].notna()]

if valid_df.empty:
    print("\n[WARN] All ICC values are NaN — summary tables not generated.")
    print("  This likely means every state had only 1 window.")
else:
    summary = (
        valid_df
        .groupby(["Group", "Mode"])["ICC_2_1"]
        .agg(
            Mean_ICC="mean",
            SD_ICC="std",
            Median_ICC="median",
            Min_ICC="min",
            Max_ICC="max",
            N_States="count",
        )
        .round(4)
        .reset_index()
    )

    summary_path = out_dir / "icc_summary_by_group.csv"
    summary.to_csv(summary_path, index=False)
    print(f"Group summary saved to:         {summary_path.resolve()}")

    print("\n" + "=" * 70)
    print("ICC(2,1) RELIABILITY SUMMARY  —  by Group and Mode")
    print("Interpretation thresholds (Koo & Mae, 2016):")
    print("  < 0.50 = Poor  |  0.50-0.75 = Moderate  |  0.75-0.90 = Good  |  > 0.90 = Excellent")
    print("=" * 70)
    print(summary.to_string(index=False))

    print("\n" + "=" * 70)
    print("ICC(2,1) RELIABILITY SUMMARY  —  per Participant (mean across states)")
    print("=" * 70)

    per_p_summary = (
        valid_df
        .groupby(["Group", "Participant", "Mode"])["ICC_2_1"]
        .agg(Mean_ICC="mean", SD_ICC="std", N_States="count")
        .round(4)
        .reset_index()
    )
    per_p_summary["Interpretation"] = per_p_summary["Mean_ICC"].apply(interpret_icc)
    print(per_p_summary.to_string(index=False))
